# RSNA Knee — rsna-knee-ens-vitk89-mlp

Notebook **self-contained** (internet off, **T4**). **Sans TTA.**

ViT-k320 : SWA epochs 8-9, FOV 130 mm.
v20b et CoAt : SWA OAI epochs 1-3, FOV 130 mm.
v20h : SWA epochs 6-8, FOV 145 mm.

- **ViT-k320** : `vit_k_llrd08_320_v3b_oofmask_vitk_full_oai3` SWA epochs 8-9 (gold in-sample moyen 0.973). Entree **320**. Pas de flip.
- **v20b DINO** : `v20b_crop130_dino_ep11_oof_v3b_full_oai3` SWA epochs 1-3 (gold in-sample moyen 0.970). Entree 224. Pas de flip.
- **v20h DINO** : `v20h_crop145_dino_ep11_oof_v3b_full_oaiv2_wd01` SWA epochs 6-8 (gold in-sample moyen 0.964). Entree 224, FOV 145 mm. Pas de flip.
- **CoAt-a** : `coat_a_llrd08_v3b_oofmask_full_oai3` SWA epochs 1-3 (gold in-sample moyen 0.988). Entree 224. Pas de flip.

**Stack MLP par cible (8 unites, GELU)** (notebook 15).
Entrees par cible : logit de chaque modele (clip 1e-4), les 6 slots vides, et le produit logit x slot.
Ajuste sur les etudes fit des predictions hybrid de ces quatre subsets.
120 epochs, log loss, L2 1e-3, Adam. Epoch retenue : meilleure AUC train (120),
train 0.9601, val 0.9617 (la val ne choisit pas l'epoch).
Pas de flip. Sans TTA. La sortie est le sigmoid du stack, pas une moyenne ponderee.


In [ ]:
import os
from pathlib import Path

KAGGLE = os.path.exists('/kaggle')
if KAGGLE:
    DATA_DIR = Path('/kaggle/input')
    print('contenu /kaggle/input :', sorted(p.name for p in DATA_DIR.iterdir()))
    COMP_DIR = None
    for base in [DATA_DIR / 'competitions', DATA_DIR]:
        if base.exists():
            for p in sorted(base.iterdir()):
                if (p / 'test.csv').exists():
                    COMP_DIR = p
                    break
        if COMP_DIR is not None:
            break
    if COMP_DIR is None:
        raise FileNotFoundError('test.csv introuvable sous /kaggle/input')
    search_roots = []
    if (DATA_DIR / 'datasets').exists():
        search_roots.append(DATA_DIR / 'datasets')
    search_roots.append(DATA_DIR)

    def _find_weight(name):
        for root in search_roots:
            hits = list(root.rglob(name))
            if hits:
                return hits[0]
        raise FileNotFoundError(name)

    WEIGHTS_VIT = _find_weight('vit_k320_vitk_oai3_swa_ep8_9.pt')
    WEIGHTS_V20H = _find_weight('v20h_crop145_swa_ep6_8.pt')
    WEIGHTS_V20B = _find_weight('v20b_dino_oai3_swa_ep1_3.pt')
    WEIGHTS_COAT = _find_weight('coat_a_oai3_swa_ep1_3.pt')
else:
    COMP_DIR = Path('../data/csv')
    WEIGHTS_VIT = Path('kaggle_dataset_vit_k/_upload_k320_vitk_oai3_swa89/vit_k320_vitk_oai3_swa_ep8_9.pt')
    WEIGHTS_V20H = Path('kaggle_dataset_v20h_lat/_upload_crop145_swa678/v20h_crop145_swa_ep6_8.pt')
    WEIGHTS_V20B = Path('kaggle_dataset_v20b/_upload_dino_oai3_swa/v20b_dino_oai3_swa_ep1_3.pt')
    WEIGHTS_COAT = Path('kaggle_dataset_coat/_upload_oai3_swa/coat_a_oai3_swa_ep1_3.pt')

TEST_CSV = COMP_DIR / 'test.csv'
TEST_SERIES_CSV = COMP_DIR / 'test_series.csv'
TEST_SERIES_DIR = COMP_DIR / 'test_series'
SAMPLE_SUB = COMP_DIR / 'sample_submission.csv'

import torch

def _pick_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        m = torch.nn.Conv2d(3, 8, 3, padding=1).cuda()
        _ = m(torch.zeros(1, 3, 32, 32, device='cuda')).sum().item()
        return 'cuda'
    except Exception as e:
        print('CUDA inutilisable, repli CPU :', str(e)[:120])
        return 'cpu'

DEVICE = _pick_device()
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

AMP_DTYPE = (torch.bfloat16
             if (DEVICE == 'cuda' and torch.cuda.is_bf16_supported())
             else torch.float16)

# Le blend lineaire est remplace par le stack charge plus bas.

print('competition dir :', COMP_DIR)
print('poids vit       :', WEIGHTS_VIT, '| existe :', WEIGHTS_VIT.exists())
print('poids v20h dino :', WEIGHTS_V20H, '| existe :', WEIGHTS_V20H.exists())
print('poids v20b dino :', WEIGHTS_V20B, '| existe :', WEIGHTS_V20B.exists())
print('poids coat      :', WEIGHTS_COAT, '| existe :', WEIGHTS_COAT.exists())
print('device          :', DEVICE)
print('TTA             : non')


In [ ]:
# --- Pretraitement DICOM -> volume (24, 224, 224) uint8 --------------------
# (copie self-contained de src/infer/preprocess_dicom.py)
# crop 130 mm + bande 20-80 % (air / bords), identique a la conversion NPZ
import cv2
import numpy as np
import pydicom

N_SLICES, IMG_SIZE = 24, 224
CROP_MM, CROP_REL, BAND = 130.0, 0.70, (0.20, 0.80)


def _slice_sort_key(ds):
    inst = getattr(ds, 'InstanceNumber', None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, 'ImagePositionPatient', None)
    iop = getattr(ds, 'ImageOrientationPatient', None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    return float(ipp[2]) if ipp is not None else 0.0


def _spacing_mm(ds):
    ps = getattr(ds, 'PixelSpacing', None) or getattr(ds, 'ImagerPixelSpacing', None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img, row_mm, col_mm):
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = min(h, max(8, int(round(CROP_MM / row_mm))))
        cw = min(w, max(8, int(round(CROP_MM / col_mm))))
    else:
        ch = min(h, max(8, int(round(h * CROP_REL))))
        cw = min(w, max(8, int(round(w * CROP_REL))))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw]


_SERIES_CACHE = {}

def load_series_dicom(series_dir, img_size=224):
    key = str(Path(series_dir))
    if key not in _SERIES_CACHE:
        _SERIES_CACHE[key] = _series_band(series_dir)
    band = _SERIES_CACHE[key]
    size = int(img_size)
    out = np.empty((band.shape[0], size, size), dtype=np.uint8)
    for i in range(band.shape[0]):
        sl = band[i]
        if sl.shape != (size, size):
            sl = cv2.resize(sl, (size, size), interpolation=cv2.INTER_AREA)
        out[i] = np.clip(np.asarray(sl, dtype=np.float32) * 255.0, 0, 255).round().astype(np.uint8)
    return out


def _series_band(series_dir):
    files = sorted(Path(series_dir).glob('*.dcm'))
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f)))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError(f'aucun DICOM lisible dans {series_dir}')
    datasets.sort(key=_slice_sort_key)

    frames, sp = [], None
    for ds in datasets:
        arr = ds.pixel_array.astype(np.float32)
        arr = arr * float(getattr(ds, 'RescaleSlope', 1.0) or 1.0) \
                  + float(getattr(ds, 'RescaleIntercept', 0.0) or 0.0)
        if getattr(ds, 'PhotometricInterpretation', 'MONOCHROME2') == 'MONOCHROME1':
            arr = arr.max() - arr
        if sp is None:
            sp = _spacing_mm(ds)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f'ndim inattendu {arr.ndim}')
    if not frames:
        raise RuntimeError('aucune coupe')

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped = [_center_crop(im, row_mm, col_mm) for im in frames]
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)
    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    band = vol[a:b + 1]
    idx = np.linspace(0, band.shape[0] - 1, N_SLICES).round().astype(int)
    band = band[idx]
    return band


PROTOCOL_SLOTS = [
    ('Sagittal', 1),
    ('Coronal', 1),
    ('Axial', 1),
    ('Sagittal', 0),
    ('Coronal', 0),
    ('Axial', 0),
]


def _pick_protocol_row(series_rows, plane, fluid):
    '''Un contraste exact (FS ou T1) sur un plan. Pas de repli.'''
    rows = series_rows[series_rows.Anatomical_Plane == plane]
    if len(rows) == 0:
        return None
    if 'Fluid_Sensitive' in rows.columns:
        rows = rows[rows.Fluid_Sensitive == int(fluid)]
    elif fluid == 1 and 'Fat_Suppression' in rows.columns:
        rows = rows[rows.Fat_Suppression == 1]
    elif fluid == 0 and 'Fat_Suppression' in rows.columns:
        rows = rows[rows.Fat_Suppression == 0]
    if len(rows) == 0:
        return None
    return rows.sort_values('SeriesInstanceUID').iloc[0]


LAT_MIN_OFFSET_MM = 20.0


def _lat_side_tag(v):
    if v is None:
        return None
    s = str(v).strip().upper()
    return s[0] if s and s[0] in ('L', 'R') else None


def _lat_center_x(ipp, iop, ps, rows, cols):
    try:
        ipp = np.asarray(list(ipp), dtype=np.float64)
        iop = np.asarray(list(iop), dtype=np.float64)
        ps = np.asarray(list(ps), dtype=np.float64)
        rows, cols = float(rows), float(cols)
    except (TypeError, ValueError):
        return None
    if ipp.size < 3 or iop.size < 6 or ps.size < 2:
        return None
    c = ipp[:3] + iop[:3] * ps[1] * cols / 2.0 + iop[3:6] * ps[0] * rows / 2.0
    x = float(c[0])
    return x if np.isfinite(x) else None


def laterality_from_study_dir(study_dir):
    '''Canon L : tag Laterality, sinon X centre (zone morte 20 mm).'''
    study_dir = Path(study_dir)
    xs, tags = [], []
    if not study_dir.is_dir():
        return None
    for series in sorted(p for p in study_dir.iterdir() if p.is_dir()):
        files = sorted(series.glob('*.dcm'))
        if not files:
            continue
        try:
            ds = pydicom.dcmread(str(files[len(files) // 2]),
                                 stop_before_pixels=True, force=True)
        except Exception:
            continue
        t = _lat_side_tag(getattr(ds, 'Laterality', None))
        if t:
            tags.append(t)
        x = _lat_center_x(
            getattr(ds, 'ImagePositionPatient', None),
            getattr(ds, 'ImageOrientationPatient', None),
            getattr(ds, 'PixelSpacing', None),
            getattr(ds, 'Rows', None), getattr(ds, 'Columns', None))
        if x is not None:
            xs.append(x)
    if tags:
        return tags[0]
    if not xs:
        return None
    m = float(np.median(xs))
    if abs(m) < LAT_MIN_OFFSET_MM:
        return None
    return 'R' if m < 0 else 'L'


def normalise_laterality(vol, plane, lat):
    if lat != 'R':
        return vol
    plane_c = str(plane or '').strip().capitalize()
    if plane_c in ('Coronal', 'Axial'):
        return np.ascontiguousarray(np.flip(vol, axis=-1))
    if plane_c == 'Sagittal':
        return np.ascontiguousarray(np.flip(vol, axis=0))
    return vol


def study_to_tensor(study_dir, series_rows, n_series=3, laterality=False, img_size=224):
    '''(n_series, 24, img_size, img_size) ; n_series=6 : 3 FS + 3 T1, sans repli.'''
    planes = ['Sagittal', 'Coronal', 'Axial']
    size = int(img_size)
    out = np.zeros((n_series, N_SLICES, size, size), dtype=np.float32)
    used = set()
    lat = laterality_from_study_dir(study_dir) if laterality else None

    def _fill(i, pick, plane):
        if pick is None:
            return
        sdir = Path(study_dir) / pick.SeriesInstanceUID
        if not sdir.exists():
            return
        try:
            vol = load_series_dicom(sdir, img_size=size).astype(np.float32) / 255.0
            if laterality:
                vol = normalise_laterality(vol, plane, lat)
            out[i] = vol
            used.add(str(pick.SeriesInstanceUID))
        except Exception as e:
            print(f'  [warn] {pick.SeriesInstanceUID}: {e}')

    if n_series == 6:
        for i, (plane, fluid) in enumerate(PROTOCOL_SLOTS):
            _fill(i, _pick_protocol_row(series_rows, plane, fluid), plane)
        return out

    for i, plane in enumerate(planes):
        rows = series_rows[series_rows.Anatomical_Plane == plane]
        if len(rows) == 0:
            continue
        fluid = rows[rows.Fluid_Sensitive == 1] if 'Fluid_Sensitive' in rows else rows.iloc[0:0]
        if len(fluid) == 0 and 'Fat_Suppression' in rows:
            fluid = rows[rows.Fat_Suppression == 1]
        pick = (fluid if len(fluid) else rows).sort_values('SeriesInstanceUID').iloc[0]
        _fill(i, pick, plane)
    if n_series >= 4:
        t1 = None
        for plane in planes:
            rows = series_rows[series_rows.Anatomical_Plane == plane]
            if 'Fluid_Sensitive' in rows:
                rows = rows[rows.Fluid_Sensitive == 0]
            rows = rows[~rows.SeriesInstanceUID.isin(used)]
            if len(rows):
                t1 = rows.sort_values('SeriesInstanceUID').iloc[0]
                break
        pl = None if t1 is None else str(t1.Anatomical_Plane)
        _fill(3, t1, pl)
    return out


In [ ]:
# --- KneeNet v20b DINO ---
# -*- coding: utf-8 -*-
"""KneeNet v20b — v17 + 6 slots protocole (FS / T1 par plan).

Base v17 inchangee : ResNet-50 RadImageNet, stem A+B, CAM 14x14 dilated,
BiLSTM residuel init 0, BN gelee. Pas de MHA (c'est v20a). Pas de gate T1.

Entree : (B, 6, 24, H, W) grayscale [0, 1]. Ordre des slots :

  0 SAG_FS   1 COR_FS   2 AX_FS
  3 SAG_T1   4 COR_T1   5 AX_T1

Chaque slot est un contraste fixe : fluide (FS) ou T1, sans repli.
Slot absent = volume zero, masque `valid` (comme un plan droppe en v17).
Le T1 n'a plus a battre le fluide dans un 4e slot unique.

Dans train_series.csv, Fluid_Sensitive == Fat_Suppression : pas de fluide
sans FS. AX_T1 remplace le slot vide SAG_FLUID_NOFS du notebook candidat.

LSTM partage sur les 6 sequences (batch = B*S). Epoch 0 recopie v16.
"""
from __future__ import annotations

from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint
from torchvision.models import resnet50

N_TARGETS = 12
DEFAULT_WEIGHTS = "weights/ResNet50.pt"
MAP_HW = 14
LSTM_HIDDEN = 256
N_PROTOCOL_SLOTS = 6


def _dilate_layer4(layer4: nn.Sequential, dilate: int = 2) -> None:
    """stride 32 -> 16, in-place. Poids 3x3 / 1x1 inchanges."""
    for i, block in enumerate(layer4):
        if i == 0:
            block.conv2.stride = (1, 1)
            if block.downsample is not None:
                block.downsample[0].stride = (1, 1)
        block.conv2.dilation = (dilate, dilate)
        block.conv2.padding = (dilate, dilate)


class _ResNet50Encoder(nn.Module):
    """ResNet-50 jusqu'a layer4 dilated. Cles `backbone.N.*` = v14/v16/v17."""

    def __init__(self) -> None:
        super().__init__()
        try:
            base = resnet50(weights=None)
        except TypeError:
            base = resnet50(pretrained=False)
        self.backbone = nn.Sequential(*list(base.children())[:-2])
        _dilate_layer4(self.backbone[7])
        self.num_features = 2048
        for m in self.modules():
            if isinstance(m, nn.ReLU):
                m.inplace = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


class KneeNetV20b(nn.Module):
    arch_id = "kneenet_v20b"

    def __init__(self, backbone: str = "resnet50", n_series: int = N_PROTOCOL_SLOTS,
                 n_targets: int = N_TARGETS, head_dim: int = 512,
                 dropout: float = 0.2, pretrained: bool = True,
                 encode_chunk: int = 16, max_depth: int = 64,
                 backbone_weights: str = DEFAULT_WEIGHTS,
                 lstm_hidden: int = LSTM_HIDDEN) -> None:
        super().__init__()
        self.n_series = n_series
        self.n_targets = n_targets
        self.encode_chunk = int(encode_chunk)
        self.backbone_name = str(backbone)
        self.map_hw = MAP_HW
        self.lstm_hidden = int(lstm_hidden)

        self.encoder = _ResNet50Encoder()
        if pretrained:
            self._load_radimagenet(backbone_weights)
        feat_dim = self.encoder.num_features

        self.plane_embed = nn.Parameter(torch.zeros(n_series, feat_dim))
        self.slice_embed = nn.Parameter(torch.zeros(max_depth, feat_dim))
        self.attn_score = nn.Linear(feat_dim, n_targets)
        self.spatial_score = nn.Conv2d(feat_dim, n_targets, kernel_size=1)
        nn.init.zeros_(self.spatial_score.weight)
        nn.init.zeros_(self.spatial_score.bias)

        self.slice_lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=self.lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.lstm_proj = nn.Linear(2 * self.lstm_hidden, feat_dim)
        nn.init.zeros_(self.lstm_proj.weight)
        nn.init.zeros_(self.lstm_proj.bias)

        self.norm = nn.LayerNorm(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim, head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_dim, 1),
        )
        self.stem_mix = nn.Conv2d(3, 3, kernel_size=1, bias=False)
        self._init_stem_mix()
        self.register_buffer(
            "img_mean", torch.zeros(1, 3, 1, 1), persistent=True,
        )
        self.register_buffer(
            "img_std", torch.ones(1, 3, 1, 1), persistent=True,
        )

    def _init_stem_mix(self) -> None:
        """out_c = in_G pour les 3 canaux -> R=G=B=k au depart."""
        with torch.no_grad():
            self.stem_mix.weight.zero_()
            self.stem_mix.weight[:, 1, 0, 0] = 1.0

    def _load_radimagenet(self, path: str) -> None:
        p = Path(path)
        if not p.is_file():
            raise FileNotFoundError(
                f"poids RadImageNet introuvables : {p.resolve()}"
            )
        sd = torch.load(p, map_location="cpu", weights_only=False)
        self.encoder.load_state_dict(sd, strict=True)
        print(
            f"RadImageNet ResNet-50 charge : {p} | CAM {MAP_HW}x{MAP_HW} "
            f"| BiLSTM hidden {self.lstm_hidden} | slots {self.n_series}",
            flush=True,
        )

    def train(self, mode: bool = True):
        super().train(mode)
        for m in self.encoder.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

    def encoder_parameters(self):
        return self.encoder.parameters()

    def head_parameters(self):
        for name, p in self.named_parameters():
            if not name.startswith("encoder."):
                yield p

    def _encode_chunks(self, rgb: torch.Tensor) -> torch.Tensor:
        """rgb : (N, 3, H, W) dans [0, 1] -> (N, C, 14, 14)."""
        n = rgb.size(0)
        c = self.encoder.num_features
        if n == 0:
            return rgb.new_zeros(0, c, MAP_HW, MAP_HW)
        outs = []
        chunk = max(1, self.encode_chunk)
        for i in range(0, n, chunk):
            part = rgb[i:i + chunk]
            if self.training:
                feats = checkpoint(self.encoder, part, use_reentrant=False)
            else:
                feats = self.encoder(part)
            outs.append(feats)
        return torch.cat(outs, dim=0)

    def _spatial_pool(self, maps: torch.Tensor, return_attn: bool = False):
        """maps (N, C, H, W) -> gap (N, C), local (N, T, C) [, attn (N, T, H, W)]."""
        n, c, h, w = maps.shape
        gap = maps.mean(dim=(-2, -1))
        logits = self.spatial_score(maps)
        attn = torch.softmax(
            logits.float().reshape(n, self.n_targets, h * w), dim=-1,
        ).to(maps.dtype)
        flat = maps.reshape(n, c, h * w)
        local = torch.einsum("nth,nch->ntc", attn, flat)
        if return_attn:
            return gap, local, attn.reshape(n, self.n_targets, h, w)
        return gap, local

    def _stem_rgb(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """Triplet A+B pour les tokens valides. x (B,S,D,H,W), idx plat."""
        _b, s, d, _h, _w = x.shape
        bs = idx // (s * d)
        rest = idx % (s * d)
        ss = rest // d
        dd = rest % d
        d_m = (dd - 1).clamp(0, d - 1)
        d_p = (dd + 1).clamp(0, d - 1)
        center = x[bs, ss, dd]
        rgb = torch.stack(
            [
                0.5 + 0.5 * (center - x[bs, ss, d_m]),
                center,
                0.5 + 0.5 * (x[bs, ss, d_p] - center),
            ],
            dim=1,
        )
        rgb = self.stem_mix(rgb)
        return (rgb - self.img_mean) / self.img_std

    def _temporal_delta(self, gaps: torch.Tensor, b: int, s: int,
                        d: int) -> torch.Tensor:
        """gaps (BSD, C) -> residu LSTM (B, S, D, C). Epoch 0 : 0."""
        c = gaps.size(-1)
        seq = gaps.reshape(b * s, d, c)
        with torch.autocast("cuda", enabled=False):
            h, _ = self.slice_lstm(seq.float())
            delta = self.lstm_proj(h)
        return delta.to(dtype=gaps.dtype).reshape(b, s, d, c)

    def _slice_pool(self, x: torch.Tensor, valid: torch.Tensor,
                    gaps: torch.Tensor, local: torch.Tensor):
        """gaps (BSD,C), local (BSD,T,C) -> logits (B,T), slice_attn (B,S,D,T)."""
        b, s, d = valid.shape
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, -1)
        feats = gaps.reshape(b, s, d, -1) + delta + pe + se
        scores = self.attn_score(feats)
        scores = scores.masked_fill(~valid.unsqueeze(-1), -1.0e4)
        scores = scores.reshape(b, s * d, self.n_targets)
        attn = torch.softmax(scores.float(), dim=1).to(feats.dtype)
        tokens = local.reshape(b, s, d, self.n_targets, -1)
        tokens = tokens + delta.unsqueeze(3) + pe.unsqueeze(3) + se.unsqueeze(3)
        tokens = tokens.reshape(b, s * d, self.n_targets, -1)
        pooled = torch.einsum("bnt,bntc->btc", attn, tokens)
        logits = self.head(self.norm(pooled)).squeeze(-1)
        return logits, attn.reshape(b, s, d, self.n_targets)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, S, D, H, W) dans [0, 1] -> logits (B, 12)."""
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        if d > self.slice_embed.size(0):
            raise ValueError(
                f"D={d} > max_depth={self.slice_embed.size(0)} : "
                "augmenter max_depth"
            )

        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            if gaps.dtype != maps.dtype:
                gaps = gaps.to(dtype=maps.dtype)
                local = local.to(dtype=maps.dtype)
            gap, loc = self._spatial_pool(maps)
            gaps[idx] = gap
            local[idx] = loc
        logits, _attn = self._slice_pool(x, valid, gaps, local)
        return logits

    @torch.no_grad()
    def interpret(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        """Forward eval + cartes. spatial_attn : (B, S, D, T, 14, 14)."""
        self.eval()
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        hw = MAP_HW
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        spatial = x.new_zeros(n_tok, self.n_targets, hw, hw)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            if gaps.dtype != maps.dtype:
                gaps = gaps.to(dtype=maps.dtype)
                local = local.to(dtype=maps.dtype)
                spatial = spatial.to(dtype=maps.dtype)
            gap, loc, spat = self._spatial_pool(maps, return_attn=True)
            gaps[idx] = gap
            local[idx] = loc
            spatial[idx] = spat
        logits, slice_attn = self._slice_pool(x, valid, gaps, local)
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, 1, -1)
        cam_tokens = (
            local.reshape(b, s, d, self.n_targets, -1)
            + delta.unsqueeze(3) + pe + se
        )
        return {
            "logits": logits,
            "slice_attn": slice_attn,
            "spatial_attn": spatial.reshape(
                b, s, d, self.n_targets, hw, hw,
            ),
            "cam_tokens": cam_tokens,
            "valid": valid,
        }


In [ ]:
# --- KneeNet v20h DINO ---
# -*- coding: utf-8 -*-
"""KneeNet v20h — v20b + tete de decision par cible (v20d, 6 slots).

Base v20b inchangee : ResNet-50 RadImageNet, stem A+B, CAM 14x14 dilated,
BiLSTM residuel init 0, BN gelee, 6 slots protocole (FS / T1 par plan).

Levier : v20b partage Linear(C->512) et Linear(512->1) sur les 12 cibles.
Ici, comme v20d (qui n'existait qu'en 3 plans) :

1. `label_embed` (T, C) init 0, ajoute a pooled avant LayerNorm
2. Linear(512 -> T) en **diagonale** : logit_t = W_t · h_t + b_t

A l'init, les T lignes de W (et les T biais) sont copies de la 1re :
epoch 0 = v20b. Les cibles peuvent ensuite diverger.

Entree : (B, 6, 24, H, W) grayscale [0, 1]. Ordre des slots = v20b.
Juger vs subset_v20b_crop130_frac025 (val ET gold, Δ > 0.01), surtout
MCL / Effusion / Medial Meniscus.
"""
from __future__ import annotations

from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint
from torchvision.models import resnet50

N_TARGETS = 12
DEFAULT_WEIGHTS = "weights/ResNet50.pt"
MAP_HW = 14
LSTM_HIDDEN = 256
N_PROTOCOL_SLOTS = 6


def _dilate_layer4(layer4: nn.Sequential, dilate: int = 2) -> None:
    """stride 32 -> 16, in-place. Poids 3x3 / 1x1 inchanges."""
    for i, block in enumerate(layer4):
        if i == 0:
            block.conv2.stride = (1, 1)
            if block.downsample is not None:
                block.downsample[0].stride = (1, 1)
        block.conv2.dilation = (dilate, dilate)
        block.conv2.padding = (dilate, dilate)


class _ResNet50Encoder(nn.Module):
    """ResNet-50 jusqu'a layer4 dilated. Cles `backbone.N.*` = v14/v16/v17."""

    def __init__(self) -> None:
        super().__init__()
        try:
            base = resnet50(weights=None)
        except TypeError:
            base = resnet50(pretrained=False)
        self.backbone = nn.Sequential(*list(base.children())[:-2])
        _dilate_layer4(self.backbone[7])
        self.num_features = 2048
        for m in self.modules():
            if isinstance(m, nn.ReLU):
                m.inplace = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


class KneeNetV20h(nn.Module):
    arch_id = "kneenet_v20h"

    def __init__(self, backbone: str = "resnet50", n_series: int = N_PROTOCOL_SLOTS,
                 n_targets: int = N_TARGETS, head_dim: int = 512,
                 dropout: float = 0.2, pretrained: bool = True,
                 encode_chunk: int = 16, max_depth: int = 64,
                 backbone_weights: str = DEFAULT_WEIGHTS,
                 lstm_hidden: int = LSTM_HIDDEN) -> None:
        super().__init__()
        self.n_series = n_series
        self.n_targets = n_targets
        self.encode_chunk = int(encode_chunk)
        self.backbone_name = str(backbone)
        self.map_hw = MAP_HW
        self.lstm_hidden = int(lstm_hidden)

        self.encoder = _ResNet50Encoder()
        if pretrained:
            self._load_radimagenet(backbone_weights)
        feat_dim = self.encoder.num_features

        self.plane_embed = nn.Parameter(torch.zeros(n_series, feat_dim))
        self.slice_embed = nn.Parameter(torch.zeros(max_depth, feat_dim))
        self.attn_score = nn.Linear(feat_dim, n_targets)
        self.spatial_score = nn.Conv2d(feat_dim, n_targets, kernel_size=1)
        nn.init.zeros_(self.spatial_score.weight)
        nn.init.zeros_(self.spatial_score.bias)

        self.slice_lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=self.lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.lstm_proj = nn.Linear(2 * self.lstm_hidden, feat_dim)
        nn.init.zeros_(self.lstm_proj.weight)
        nn.init.zeros_(self.lstm_proj.bias)

        self.label_embed = nn.Parameter(torch.zeros(n_targets, feat_dim))
        self.norm = nn.LayerNorm(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim, head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.head_out = nn.Linear(head_dim, n_targets)
        with torch.no_grad():
            w0 = self.head_out.weight[0].detach().clone()
            b0 = self.head_out.bias[0].detach().clone()
            self.head_out.weight.copy_(w0.unsqueeze(0).expand_as(self.head_out.weight))
            self.head_out.bias.copy_(b0.expand_as(self.head_out.bias))

        self.stem_mix = nn.Conv2d(3, 3, kernel_size=1, bias=False)
        self._init_stem_mix()
        self.register_buffer(
            "img_mean", torch.zeros(1, 3, 1, 1), persistent=True,
        )
        self.register_buffer(
            "img_std", torch.ones(1, 3, 1, 1), persistent=True,
        )

    def extra_log(self) -> dict[str, float]:
        """Divergence des lignes de head_out vs la 1re (init liee = 0)."""
        w = self.head_out.weight.detach()
        b = self.head_out.bias.detach()
        dw = (w - w[0]).abs().mean()
        db = (b - b[0]).abs().mean()
        return {
            "head_out_w_div": float(dw),
            "head_out_b_div": float(db),
            "label_embed_abs": float(self.label_embed.detach().abs().mean()),
        }

    def _init_stem_mix(self) -> None:
        """out_c = in_G pour les 3 canaux -> R=G=B=k au depart."""
        with torch.no_grad():
            self.stem_mix.weight.zero_()
            self.stem_mix.weight[:, 1, 0, 0] = 1.0

    def _load_radimagenet(self, path: str) -> None:
        p = Path(path)
        if not p.is_file():
            raise FileNotFoundError(
                f"poids RadImageNet introuvables : {p.resolve()}"
            )
        sd = torch.load(p, map_location="cpu", weights_only=False)
        self.encoder.load_state_dict(sd, strict=True)
        print(
            f"RadImageNet ResNet-50 charge : {p} | CAM {MAP_HW}x{MAP_HW} "
            f"| BiLSTM hidden {self.lstm_hidden} | slots {self.n_series} "
            f"| tete par cible (diag, init liee)",
            flush=True,
        )

    def train(self, mode: bool = True):
        super().train(mode)
        for m in self.encoder.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

    def encoder_parameters(self):
        return self.encoder.parameters()

    def head_parameters(self):
        for name, p in self.named_parameters():
            if not name.startswith("encoder."):
                yield p

    def _encode_chunks(self, rgb: torch.Tensor) -> torch.Tensor:
        """rgb : (N, 3, H, W) dans [0, 1] -> (N, C, 14, 14)."""
        n = rgb.size(0)
        c = self.encoder.num_features
        if n == 0:
            return rgb.new_zeros(0, c, MAP_HW, MAP_HW)
        outs = []
        chunk = max(1, self.encode_chunk)
        for i in range(0, n, chunk):
            part = rgb[i:i + chunk]
            if self.training:
                feats = checkpoint(self.encoder, part, use_reentrant=False)
            else:
                feats = self.encoder(part)
            outs.append(feats)
        return torch.cat(outs, dim=0)

    def _spatial_pool(self, maps: torch.Tensor, return_attn: bool = False):
        """maps (N, C, H, W) -> gap (N, C), local (N, T, C) [, attn (N, T, H, W)]."""
        n, c, h, w = maps.shape
        gap = maps.mean(dim=(-2, -1))
        logits = self.spatial_score(maps)
        attn = torch.softmax(
            logits.float().reshape(n, self.n_targets, h * w), dim=-1,
        ).to(maps.dtype)
        flat = maps.reshape(n, c, h * w)
        local = torch.einsum("nth,nch->ntc", attn, flat)
        if return_attn:
            return gap, local, attn.reshape(n, self.n_targets, h, w)
        return gap, local

    def _stem_rgb(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """Triplet A+B pour les tokens valides. x (B,S,D,H,W), idx plat."""
        _b, s, d, _h, _w = x.shape
        bs = idx // (s * d)
        rest = idx % (s * d)
        ss = rest // d
        dd = rest % d
        d_m = (dd - 1).clamp(0, d - 1)
        d_p = (dd + 1).clamp(0, d - 1)
        center = x[bs, ss, dd]
        rgb = torch.stack(
            [
                0.5 + 0.5 * (center - x[bs, ss, d_m]),
                center,
                0.5 + 0.5 * (x[bs, ss, d_p] - center),
            ],
            dim=1,
        )
        rgb = self.stem_mix(rgb)
        return (rgb - self.img_mean) / self.img_std

    def _temporal_delta(self, gaps: torch.Tensor, b: int, s: int,
                        d: int) -> torch.Tensor:
        """gaps (BSD, C) -> residu LSTM (B, S, D, C). Epoch 0 : 0."""
        c = gaps.size(-1)
        seq = gaps.reshape(b * s, d, c)
        with torch.autocast("cuda", enabled=False):
            h, _ = self.slice_lstm(seq.float())
            delta = self.lstm_proj(h)
        return delta.to(dtype=gaps.dtype).reshape(b, s, d, c)

    def _head_logits(self, pooled: torch.Tensor) -> torch.Tensor:
        """pooled (B, T, C) -> logits (B, T). Epoch 0 = tete v20b partagee."""
        pooled = pooled + self.label_embed.to(dtype=pooled.dtype)
        hidden = self.head(self.norm(pooled))
        return self.head_out(hidden).diagonal(dim1=-2, dim2=-1)

    def _slice_pool(self, x: torch.Tensor, valid: torch.Tensor,
                    gaps: torch.Tensor, local: torch.Tensor):
        """gaps (BSD,C), local (BSD,T,C) -> logits (B,T), slice_attn (B,S,D,T)."""
        b, s, d = valid.shape
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, -1)
        feats = gaps.reshape(b, s, d, -1) + delta + pe + se
        scores = self.attn_score(feats)
        scores = scores.masked_fill(~valid.unsqueeze(-1), -1.0e4)
        scores = scores.reshape(b, s * d, self.n_targets)
        attn = torch.softmax(scores.float(), dim=1).to(feats.dtype)
        tokens = local.reshape(b, s, d, self.n_targets, -1)
        tokens = tokens + delta.unsqueeze(3) + pe.unsqueeze(3) + se.unsqueeze(3)
        tokens = tokens.reshape(b, s * d, self.n_targets, -1)
        pooled = torch.einsum("bnt,bntc->btc", attn, tokens)
        logits = self._head_logits(pooled)
        return logits, attn.reshape(b, s, d, self.n_targets)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, S, D, H, W) dans [0, 1] -> logits (B, 12)."""
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        if d > self.slice_embed.size(0):
            raise ValueError(
                f"D={d} > max_depth={self.slice_embed.size(0)} : "
                "augmenter max_depth"
            )

        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            if gaps.dtype != maps.dtype:
                gaps = gaps.to(dtype=maps.dtype)
                local = local.to(dtype=maps.dtype)
            gap, loc = self._spatial_pool(maps)
            gaps[idx] = gap
            local[idx] = loc
        logits, _attn = self._slice_pool(x, valid, gaps, local)
        return logits

    @torch.no_grad()
    def interpret(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        """Forward eval + cartes. spatial_attn : (B, S, D, T, 14, 14)."""
        self.eval()
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        hw = MAP_HW
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        spatial = x.new_zeros(n_tok, self.n_targets, hw, hw)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            if gaps.dtype != maps.dtype:
                gaps = gaps.to(dtype=maps.dtype)
                local = local.to(dtype=maps.dtype)
                spatial = spatial.to(dtype=maps.dtype)
            gap, loc, spat = self._spatial_pool(maps, return_attn=True)
            gaps[idx] = gap
            local[idx] = loc
            spatial[idx] = spat
        logits, slice_attn = self._slice_pool(x, valid, gaps, local)
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, 1, -1)
        cam_tokens = (
            local.reshape(b, s, d, self.n_targets, -1)
            + delta.unsqueeze(3) + pe + se
        )
        return {
            "logits": logits,
            "slice_attn": slice_attn,
            "spatial_attn": spatial.reshape(
                b, s, d, self.n_targets, hw, hw,
            ),
            "cam_tokens": cam_tokens,
            "valid": valid,
        }


In [ ]:
# --- KneeNet ViT-e, poids ViT-k320 ---
# -*- coding: utf-8 -*-
"""KneeNet-ViT-e — v1 (CAM+BiLSTM) + stem RGB k-1/k/k+1 de d.

Famille CLS (c/d) : le 2.5D aide (contusion 0.66 -> 0.72) mais le gold
reste ~0.74 vs v1 0.80. Le plafond CLS+v8 est sous v1. On recolle le
graphe qui score (CAM patches + LSTM) et on y met le seul levier D
qui a marche.

  - stem : R=k-1, G=k, B=k+1, stem_mix init G (epoch 0 == v1)
  - cartes patch 14x14, CAM + GAP, BiLSTM init 0 (comme v1)
  - LayerScale gele (recette c/d)

Recette : freeze 1, lr_backbone 3e-5 (v1 tenait), re-warmup 1 epoch.
Option `drop_path_rate` (YAML) : stochastic depth timm, OFF pendant
le freeze (epoch 1 = identity e). Defaut 0 = graphe e inchange.
Poids : `weights/dinov3-vitb16-lvd1689m/model.safetensors`
Entree : (B, 6, 24, H, W) grayscale [0, 1]. Slots = v20b.
"""
from __future__ import annotations

from pathlib import Path

import timm
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

N_TARGETS = 12
DEFAULT_WEIGHTS = "weights/dinov3-vitb16-lvd1689m"
LSTM_HIDDEN = 256
N_PROTOCOL_SLOTS = 6
PATCH_SIZE = 16
N_PREFIX = 5
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
TIMM_NAME = "vit_base_patch16_dinov3.lvd1689m"


def _weights_file(path: str | Path) -> Path:
    p = Path(path)
    if p.is_dir():
        for name in ("model.safetensors", "pytorch_model.bin"):
            cand = p / name
            if cand.is_file():
                return cand
        raise FileNotFoundError(
            f"aucun checkpoint dans {p.resolve()} "
            "(attendu model.safetensors)"
        )
    if not p.is_file():
        raise FileNotFoundError(f"poids DINOv3 introuvables : {p.resolve()}")
    return p


class _Dinov3VitEncoder(nn.Module):
    """ViT-B/16 DINOv3 -> cartes (N, 768, H/16, W/16)."""

    def __init__(self, weights_path: str = DEFAULT_WEIGHTS,
                 pretrained: bool = True,
                 drop_path_rate: float = 0.0,
                 grad_checkpoint: bool = True) -> None:
        super().__init__()
        kwargs = dict(
            num_classes=0,
            global_pool="",
            dynamic_img_size=True,
        )
        dp = float(drop_path_rate or 0.0)
        if dp > 0:
            kwargs["drop_path_rate"] = dp
        if pretrained:
            ckpt = _weights_file(weights_path)
            if ckpt.suffix == ".pt":
                self.backbone = timm.create_model(
                    TIMM_NAME, pretrained=False, **kwargs,
                )
                sd = torch.load(ckpt, map_location="cpu", weights_only=False)
                self.load_state_dict(sd, strict=True)
                print(
                    f"DINOv3 ViT-B/16 charge : {ckpt} | {len(sd)} tenseurs",
                    flush=True,
                )
            else:
                self.backbone = timm.create_model(
                    TIMM_NAME,
                    pretrained=True,
                    pretrained_cfg_overlay={"file": str(ckpt)},
                    **kwargs,
                )
                print(f"DINOv3 ViT-B/16 charge : {ckpt}", flush=True)
        else:
            self.backbone = timm.create_model(
                TIMM_NAME, pretrained=False, **kwargs,
            )
        self.backbone.set_grad_checkpointing(bool(grad_checkpoint))
        self.num_features = int(self.backbone.num_features)
        self.patch_size = PATCH_SIZE
        self.n_prefix = int(getattr(
            self.backbone, "num_prefix_tokens", N_PREFIX))
        self.drop_path_rate = dp
        self._drop_path: list[tuple[nn.Module, float]] = []
        for m in self.backbone.modules():
            if m.__class__.__name__ == "DropPath":
                self._drop_path.append((m, float(m.drop_prob)))
        if dp > 0:
            print(
                f"drop_path {dp:g} | {len(self._drop_path)} modules "
                f"(off pendant freeze)",
                flush=True,
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x (N, 3, H, W) -> (N, C, h, w) patch tokens."""
        tokens = self.backbone.forward_features(x)
        patches = tokens[:, self.n_prefix:]
        n, n_hw, c = patches.shape
        side = int(n_hw ** 0.5)
        if side * side != n_hw:
            raise ValueError(
                f"grille patch non carree : {n_hw} tokens "
                f"(H={x.shape[-2]}, W={x.shape[-1]})"
            )
        return patches.transpose(1, 2).reshape(n, c, side, side)


class KneeNetVitE(nn.Module):
    arch_id = "kneenet_vit_e"

    def __init__(self, backbone: str = TIMM_NAME,
                 n_series: int = N_PROTOCOL_SLOTS,
                 n_targets: int = N_TARGETS, head_dim: int = 512,
                 dropout: float = 0.2, pretrained: bool = True,
                 encode_chunk: int = 4, max_depth: int = 64,
                 backbone_weights: str = DEFAULT_WEIGHTS,
                 lstm_hidden: int = LSTM_HIDDEN,
                 drop_path_rate: float = 0.0,
                 grad_checkpoint: bool = True) -> None:
        super().__init__()
        self.n_series = n_series
        self.n_targets = n_targets
        self.encode_chunk = int(encode_chunk)
        # False sur grosse carte : sinon le ViT est decoupe en petits
        # lancements et le GPU attend Python entre deux blocs.
        self.grad_checkpoint = bool(grad_checkpoint)
        self.backbone_name = str(backbone)
        self.patch_size = PATCH_SIZE
        self.lstm_hidden = int(lstm_hidden)
        self.drop_path_rate = float(drop_path_rate or 0.0)

        self.encoder = _Dinov3VitEncoder(
            weights_path=backbone_weights, pretrained=pretrained,
            drop_path_rate=self.drop_path_rate,
            grad_checkpoint=self.grad_checkpoint,
        )
        self._freeze_layer_scale()
        feat_dim = self.encoder.num_features

        self.stem_mix = nn.Conv2d(3, 3, kernel_size=1, bias=False)
        self._init_stem_mix()

        self.plane_embed = nn.Parameter(torch.zeros(n_series, feat_dim))
        self.slice_embed = nn.Parameter(torch.zeros(max_depth, feat_dim))
        self.attn_score = nn.Linear(feat_dim, n_targets)
        self.spatial_score = nn.Conv2d(feat_dim, n_targets, kernel_size=1)
        nn.init.zeros_(self.spatial_score.weight)
        nn.init.zeros_(self.spatial_score.bias)

        self.slice_lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=self.lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.lstm_proj = nn.Linear(2 * self.lstm_hidden, feat_dim)
        nn.init.zeros_(self.lstm_proj.weight)
        nn.init.zeros_(self.lstm_proj.bias)

        self.norm = nn.LayerNorm(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim, head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_dim, 1),
        )
        self.register_buffer(
            "img_mean",
            torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1),
            persistent=True,
        )
        self.register_buffer(
            "img_std",
            torch.tensor(IMAGENET_STD).view(1, 3, 1, 1),
            persistent=True,
        )
        dp_txt = (f" | drop_path {self.drop_path_rate:g}"
                  if self.drop_path_rate > 0 else "")
        print(
            f"KneeNet-ViT-e | CAM patch {PATCH_SIZE} | BiLSTM "
            f"| RGB k-1/k/k+1{dp_txt} | dim {feat_dim} | slots {self.n_series}",
            flush=True,
        )

    def _init_stem_mix(self) -> None:
        """out = in_G -> epoch 0 identique a v1."""
        with torch.no_grad():
            self.stem_mix.weight.zero_()
            self.stem_mix.weight[:, 1, 0, 0] = 1.0

    @staticmethod
    def _is_layerscale(name: str) -> bool:
        return name.rsplit(".", 1)[-1].startswith("gamma")

    def _freeze_layer_scale(self) -> None:
        n = 0
        for name, p in self.encoder.named_parameters():
            if self._is_layerscale(name):
                p.requires_grad = False
                n += 1
        print(f"LayerScale gele : {n} tenseurs gamma", flush=True)

    def set_encoder_frozen(self, flag: bool) -> None:
        for name, p in self.encoder.named_parameters():
            p.requires_grad = (not flag) and not self._is_layerscale(name)
        self.stem_mix.weight.requires_grad = not flag
        # DropPath stochastique seulement une fois l'encodeur degele,
        # sinon l'epoch 1 n'est plus identity-e.
        for m, p0 in self.encoder._drop_path:
            m.drop_prob = 0.0 if flag else p0

    def encoder_parameters(self):
        for p in self.encoder.parameters():
            if p.requires_grad:
                yield p
        if self.stem_mix.weight.requires_grad:
            yield self.stem_mix.weight

    def head_parameters(self):
        for name, p in self.named_parameters():
            if name.startswith("encoder.") or name.startswith("stem_mix."):
                continue
            yield p

    def _encode_chunks(self, rgb: torch.Tensor) -> torch.Tensor:
        n = rgb.size(0)
        c = self.encoder.num_features
        if n == 0:
            h = rgb.size(-2) // self.patch_size
            w = rgb.size(-1) // self.patch_size
            return rgb.new_zeros(0, c, h, w)
        outs = []
        chunk = max(1, self.encode_chunk)
        for i in range(0, n, chunk):
            part = rgb[i:i + chunk]
            if self.training and self.grad_checkpoint:
                feats = checkpoint(self.encoder, part, use_reentrant=False)
            else:
                feats = self.encoder(part)
            outs.append(feats)
        return torch.cat(outs, dim=0)

    def _spatial_pool(self, maps: torch.Tensor, return_attn: bool = False):
        n, c, h, w = maps.shape
        gap = maps.mean(dim=(-2, -1))
        logits = self.spatial_score(maps)
        attn = torch.softmax(
            logits.float().reshape(n, self.n_targets, h * w), dim=-1,
        ).to(maps.dtype)
        flat = maps.reshape(n, c, h * w)
        local = torch.einsum("nth,nch->ntc", attn, flat)
        if return_attn:
            return gap, local, attn.reshape(n, self.n_targets, h, w)
        return gap, local

    def _stem_rgb(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """R=k-1, G=k, B=k+1 puis stem_mix. x (B,S,D,H,W), idx plat."""
        _b, s, d, _h, _w = x.shape
        bs = idx // (s * d)
        rest = idx % (s * d)
        ss = rest // d
        dd = rest % d
        d_m = (dd - 1).clamp(0, d - 1)
        d_p = (dd + 1).clamp(0, d - 1)
        rgb = torch.stack(
            [x[bs, ss, d_m], x[bs, ss, dd], x[bs, ss, d_p]],
            dim=1,
        )
        rgb = self.stem_mix(rgb)
        return (rgb - self.img_mean) / self.img_std

    def _temporal_delta(self, gaps: torch.Tensor, b: int, s: int,
                        d: int) -> torch.Tensor:
        c = gaps.size(-1)
        seq = gaps.reshape(b * s, d, c)
        with torch.autocast("cuda", enabled=False):
            h, _ = self.slice_lstm(seq.float())
            delta = self.lstm_proj(h)
        return delta.to(dtype=gaps.dtype).reshape(b, s, d, c)

    def _slice_pool(self, x: torch.Tensor, valid: torch.Tensor,
                    gaps: torch.Tensor, local: torch.Tensor):
        b, s, d = valid.shape
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, -1)
        feats = gaps.reshape(b, s, d, -1) + delta + pe + se
        scores = self.attn_score(feats)
        scores = scores.masked_fill(~valid.unsqueeze(-1), -1.0e4)
        scores = scores.reshape(b, s * d, self.n_targets)
        attn = torch.softmax(scores.float(), dim=1).to(feats.dtype)
        tokens = local.reshape(b, s, d, self.n_targets, -1)
        tokens = tokens + delta.unsqueeze(3) + pe.unsqueeze(3) + se.unsqueeze(3)
        tokens = tokens.reshape(b, s * d, self.n_targets, -1)
        pooled = torch.einsum("bnt,bntc->btc", attn, tokens)
        logits = self.head(self.norm(pooled)).squeeze(-1)
        return logits, attn.reshape(b, s, d, self.n_targets)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        if d > self.slice_embed.size(0):
            raise ValueError(
                f"D={d} > max_depth={self.slice_embed.size(0)} : "
                "augmenter max_depth"
            )
        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            gap, loc = self._spatial_pool(maps)
            if gaps.dtype != gap.dtype:
                gaps = gaps.to(dtype=gap.dtype)
            if local.dtype != loc.dtype:
                local = local.to(dtype=loc.dtype)
            gaps[idx] = gap
            local[idx] = loc
        logits, _attn = self._slice_pool(x, valid, gaps, local)
        return logits

    @torch.no_grad()
    def interpret(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        self.eval()
        b, s, d, img_h, img_w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        hw_h, hw_w = img_h // self.patch_size, img_w // self.patch_size
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        spatial = x.new_zeros(n_tok, self.n_targets, hw_h, hw_w)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            gap, loc, spat = self._spatial_pool(maps, return_attn=True)
            if gaps.dtype != gap.dtype:
                gaps = gaps.to(dtype=gap.dtype)
            if local.dtype != loc.dtype:
                local = local.to(dtype=loc.dtype)
            if spatial.dtype != spat.dtype:
                spatial = spatial.to(dtype=spat.dtype)
            gaps[idx] = gap
            local[idx] = loc
            spatial[idx] = spat
        logits, slice_attn = self._slice_pool(x, valid, gaps, local)
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, 1, -1)
        cam_tokens = (
            local.reshape(b, s, d, self.n_targets, -1)
            + delta.unsqueeze(3) + pe + se
        )
        return {
            "logits": logits,
            "slice_attn": slice_attn,
            "spatial_attn": spatial.reshape(
                b, s, d, self.n_targets, hw_h, hw_w,
            ),
            "cam_tokens": cam_tokens,
            "valid": valid,
        }


In [ ]:
# --- KneeNet CoAt-a ---
# -*- coding: utf-8 -*-
"""KneeNet-CoAt-a — baseline CoAtNet 2.5D @224, graphe ViT-e / v20b.

Seul l'encodeur change vs ViT-e : CoAtNet-0 timm (stem conv + 2 stages
MBConv BN + 2 stages Transformer LN), ImageNet-1k, 27M params (~ResNet-50).
Le reste est le graphe qui score deja :

  - stem 2.5D : R=k-1, G=k, B=k+1 (voisins linspace crop130), stem_mix
    1x1 init G -> epoch 0 = coupe grise repliquee (in-distribution)
  - carte finale 7x7 (stride 32 @224) : CAM 12 tetes + GAP
  - BiLSTM profondeur residuel init 0, attention coupe par cible, 6 slots
  - BN encodeur gelee (running stats ImageNet, affine entrainable),
    comme v20b : chunks correles + checkpoint = stats instables

Normalisation = pretrained_cfg timm (0.5 / 0.5 pour les CoAtNet rw),
pas ImageNet. `backbone` YAML = autre CoAtNet timm (poids assortis).
Entree : (B, 6, 24, H, W) grayscale [0, 1]. Slots = v20b.
"""
from __future__ import annotations

from pathlib import Path

import timm
print('timm', timm.__version__, flush=True)
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

COAT_N_TARGETS = 12
COAT_TIMM_NAME = "coatnet_0_rw_224.sw_in1k"
COAT_DEFAULT_WEIGHTS = "weights/coatnet_0_rw_224-sw_in1k"
COAT_LSTM_HIDDEN = 256
COAT_N_PROTOCOL_SLOTS = 6


def _coat_weights_file(path: str | Path) -> Path:
    p = Path(path)
    if p.is_dir():
        for name in ("model.safetensors", "pytorch_model.bin"):
            cand = p / name
            if cand.is_file():
                return cand
        raise FileNotFoundError(
            f"aucun checkpoint dans {p.resolve()} "
            "(attendu model.safetensors)"
        )
    if not p.is_file():
        raise FileNotFoundError(f"poids CoAtNet introuvables : {p.resolve()}")
    return p


class _CoatEncoder(nn.Module):
    """CoAtNet timm -> cartes (N, C, H/32, W/32) apres LayerNorm finale."""

    def __init__(self, name: str = COAT_TIMM_NAME,
                 weights_path: str = COAT_DEFAULT_WEIGHTS,
                 pretrained: bool = True,
                 img_size: int = 224) -> None:
        super().__init__()
        self.img_size = int(img_size)
        kwargs = dict(num_classes=0, global_pool="", img_size=self.img_size)
        if pretrained:
            ckpt = _coat_weights_file(weights_path)
            self.backbone = timm.create_model(
                name, pretrained=True,
                pretrained_cfg_overlay={"file": str(ckpt)}, **kwargs,
            )
            print(f"CoAtNet charge : {name} | {ckpt} | entree {self.img_size}",
                  flush=True)
        else:
            self.backbone = timm.create_model(name, pretrained=False, **kwargs)
        self.num_features = int(self.backbone.num_features)
        cfg = self.backbone.pretrained_cfg
        self.mean = tuple(float(v) for v in cfg.get("mean", (0.5, 0.5, 0.5)))
        self.std = tuple(float(v) for v in cfg.get("std", (0.5, 0.5, 0.5)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone.forward_features(x)


class KneeNetCoat(nn.Module):
    arch_id = "kneenet_coat_a"

    def __init__(self, backbone: str = COAT_TIMM_NAME,
                 n_series: int = COAT_N_PROTOCOL_SLOTS,
                 n_targets: int = COAT_N_TARGETS, head_dim: int = 512,
                 dropout: float = 0.2, pretrained: bool = True,
                 encode_chunk: int = 8, max_depth: int = 64,
                 backbone_weights: str = COAT_DEFAULT_WEIGHTS,
                 lstm_hidden: int = COAT_LSTM_HIDDEN,
                 img_size: int = 224) -> None:
        super().__init__()
        self.n_series = n_series
        self.n_targets = n_targets
        self.encode_chunk = int(encode_chunk)
        self.backbone_name = str(backbone)
        self.lstm_hidden = int(lstm_hidden)
        self.img_size = int(img_size)

        self.encoder = _CoatEncoder(
            name=self.backbone_name, weights_path=backbone_weights,
            pretrained=pretrained, img_size=self.img_size,
        )
        feat_dim = self.encoder.num_features

        self.stem_mix = nn.Conv2d(3, 3, kernel_size=1, bias=False)
        self._init_stem_mix()

        self.plane_embed = nn.Parameter(torch.zeros(n_series, feat_dim))
        self.slice_embed = nn.Parameter(torch.zeros(max_depth, feat_dim))
        self.attn_score = nn.Linear(feat_dim, n_targets)
        self.spatial_score = nn.Conv2d(feat_dim, n_targets, kernel_size=1)
        nn.init.zeros_(self.spatial_score.weight)
        nn.init.zeros_(self.spatial_score.bias)

        self.slice_lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=self.lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.lstm_proj = nn.Linear(2 * self.lstm_hidden, feat_dim)
        nn.init.zeros_(self.lstm_proj.weight)
        nn.init.zeros_(self.lstm_proj.bias)

        self.norm = nn.LayerNorm(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim, head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_dim, 1),
        )
        self.register_buffer(
            "img_mean", torch.tensor(self.encoder.mean).view(1, 3, 1, 1),
            persistent=True,
        )
        self.register_buffer(
            "img_std", torch.tensor(self.encoder.std).view(1, 3, 1, 1),
            persistent=True,
        )
        print(
            f"KneeNet-CoAt-a | {self.backbone_name} | CAM + BiLSTM "
            f"| RGB k-1/k/k+1 | dim {feat_dim} | slots {self.n_series} "
            f"| norm {self.encoder.mean}/{self.encoder.std}",
            flush=True,
        )

    def _init_stem_mix(self) -> None:
        with torch.no_grad():
            self.stem_mix.weight.zero_()
            self.stem_mix.weight[:, 1, 0, 0] = 1.0

    def train(self, mode: bool = True):
        super().train(mode)
        for m in self.encoder.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

    def set_encoder_frozen(self, flag: bool) -> None:
        for p in self.encoder.parameters():
            p.requires_grad = not flag
        self.stem_mix.weight.requires_grad = not flag

    def encoder_parameters(self):
        for p in self.encoder.parameters():
            if p.requires_grad:
                yield p
        if self.stem_mix.weight.requires_grad:
            yield self.stem_mix.weight

    def head_parameters(self):
        for name, p in self.named_parameters():
            if name.startswith("encoder.") or name.startswith("stem_mix."):
                continue
            yield p

    @torch.no_grad()
    def extra_log(self) -> dict[str, float]:
        """Part de |stem_mix| sur chaque coupe d'entree (G seul = pas de 2.5D)."""
        w = self.stem_mix.weight.detach().abs().sum(dim=(0, 2, 3))
        w = w / w.sum().clamp(min=1e-8)
        return {"stem_mix_km1": float(w[0]), "stem_mix_k": float(w[1]),
                "stem_mix_kp1": float(w[2])}

    def _encode_chunks(self, rgb: torch.Tensor) -> torch.Tensor:
        n = rgb.size(0)
        if n == 0:
            h, w = rgb.size(-2) // 32, rgb.size(-1) // 32
            return rgb.new_zeros(0, self.encoder.num_features, h, w)
        outs = []
        chunk = max(1, self.encode_chunk)
        for i in range(0, n, chunk):
            part = rgb[i:i + chunk]
            if self.training:
                feats = checkpoint(self.encoder, part, use_reentrant=False)
            else:
                feats = self.encoder(part)
            outs.append(feats)
        return torch.cat(outs, dim=0)

    def _spatial_pool(self, maps: torch.Tensor, return_attn: bool = False):
        """maps (N, C, H, W) -> gap (N, C), local (N, T, C) [, attn (N, T, H, W)]."""
        n, c, h, w = maps.shape
        gap = maps.mean(dim=(-2, -1))
        logits = self.spatial_score(maps)
        attn = torch.softmax(
            logits.float().reshape(n, self.n_targets, h * w), dim=-1,
        ).to(maps.dtype)
        flat = maps.reshape(n, c, h * w)
        local = torch.einsum("nth,nch->ntc", attn, flat)
        if return_attn:
            return gap, local, attn.reshape(n, self.n_targets, h, w)
        return gap, local

    def _stem_rgb(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """R=k-1, G=k, B=k+1 puis stem_mix. x (B,S,D,H,W), idx plat."""
        _b, s, d, _h, _w = x.shape
        bs = idx // (s * d)
        rest = idx % (s * d)
        ss = rest // d
        dd = rest % d
        d_m = (dd - 1).clamp(0, d - 1)
        d_p = (dd + 1).clamp(0, d - 1)
        rgb = torch.stack(
            [x[bs, ss, d_m], x[bs, ss, dd], x[bs, ss, d_p]],
            dim=1,
        )
        rgb = self.stem_mix(rgb)
        return (rgb - self.img_mean) / self.img_std

    def _temporal_delta(self, gaps: torch.Tensor, b: int, s: int,
                        d: int) -> torch.Tensor:
        c = gaps.size(-1)
        seq = gaps.reshape(b * s, d, c)
        with torch.autocast("cuda", enabled=False):
            h, _ = self.slice_lstm(seq.float())
            delta = self.lstm_proj(h)
        return delta.to(dtype=gaps.dtype).reshape(b, s, d, c)

    def _slice_pool(self, x: torch.Tensor, valid: torch.Tensor,
                    gaps: torch.Tensor, local: torch.Tensor):
        b, s, d = valid.shape
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, -1)
        feats = gaps.reshape(b, s, d, -1) + delta + pe + se
        scores = self.attn_score(feats)
        scores = scores.masked_fill(~valid.unsqueeze(-1), -1.0e4)
        scores = scores.reshape(b, s * d, self.n_targets)
        attn = torch.softmax(scores.float(), dim=1).to(feats.dtype)
        tokens = local.reshape(b, s, d, self.n_targets, -1)
        tokens = tokens + delta.unsqueeze(3) + pe.unsqueeze(3) + se.unsqueeze(3)
        tokens = tokens.reshape(b, s * d, self.n_targets, -1)
        pooled = torch.einsum("bnt,bntc->btc", attn, tokens)
        logits = self.head(self.norm(pooled)).squeeze(-1)
        return logits, attn.reshape(b, s, d, self.n_targets)

    def _encode_valid(self, x: torch.Tensor, valid: torch.Tensor,
                      return_attn: bool = False):
        b, s, d, img_h, img_w = x.shape
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        spatial = None
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            pooled = self._spatial_pool(maps, return_attn=return_attn)
            gap, loc = pooled[0], pooled[1]
            gaps = gaps.to(dtype=gap.dtype)
            local = local.to(dtype=loc.dtype)
            gaps[idx] = gap
            local[idx] = loc
            if return_attn:
                spat = pooled[2]
                spatial = spat.new_zeros(n_tok, self.n_targets, *spat.shape[-2:])
                spatial[idx] = spat
        if return_attn and spatial is None:
            spatial = x.new_zeros(n_tok, self.n_targets, img_h // 32, img_w // 32)
        return gaps, local, spatial

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, S, D, H, W) dans [0, 1] -> logits (B, 12)."""
        _b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        if d > self.slice_embed.size(0):
            raise ValueError(
                f"D={d} > max_depth={self.slice_embed.size(0)} : "
                "augmenter max_depth"
            )
        valid = x.amax(dim=(-2, -1)) > 1e-6
        gaps, local, _ = self._encode_valid(x, valid)
        logits, _attn = self._slice_pool(x, valid, gaps, local)
        return logits

    @torch.no_grad()
    def interpret(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        """Forward eval + cartes. spatial_attn : (B, S, D, T, h, w)."""
        self.eval()
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        valid = x.amax(dim=(-2, -1)) > 1e-6
        gaps, local, spatial = self._encode_valid(x, valid, return_attn=True)
        logits, slice_attn = self._slice_pool(x, valid, gaps, local)
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, 1, -1)
        cam_tokens = (
            local.reshape(b, s, d, self.n_targets, -1)
            + delta.unsqueeze(3) + pe + se
        )
        hw_h, hw_w = spatial.shape[-2:]
        return {
            "logits": logits,
            "slice_attn": slice_attn,
            "spatial_attn": spatial.reshape(b, s, d, self.n_targets, hw_h, hw_w),
            "cam_tokens": cam_tokens,
            "valid": valid,
        }


In [ ]:
import pandas as pd
import time

TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
           'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
           'Contusion', 'Fracture']


def _cap_chunk(cfg, cap):
    raw = int(cfg.get('encode_chunk', cap))
    chunk = cap if raw > cap else raw
    if chunk != raw:
        print(f'encode_chunk {raw} -> {chunk} (T4)')
    return chunk


def _expect_swa(ckpt, label, epochs):
    if str(ckpt.get('epoch')) != label:
        raise RuntimeError(ckpt.get('epoch'))
    if [int(e) for e in (ckpt.get('swa_epochs') or [])] != list(epochs):
        raise RuntimeError(ckpt.get('swa_epochs'))


def load_vit(weights_path):
    ckpt = torch.load(str(weights_path), map_location='cpu', weights_only=False)
    cfg = ckpt.get('config', {})
    if str(cfg.get('run_name')) != 'vit_k_llrd08_320_v3b_oofmask_vitk_full_oai3':
        raise RuntimeError(cfg.get('run_name'))
    if str(cfg.get('model')) != 'kneenet_vit_k320':
        raise RuntimeError(cfg.get('model'))
    if int(cfg.get('img_size', 0)) != 320:
        raise RuntimeError(cfg.get('img_size'))
    if cfg.get('laterality'):
        raise RuntimeError('vit-k320 ne doit pas etre flipped')
    _expect_swa(ckpt, 'swa-8-9', [8, 9])
    model = KneeNetVitE(
        n_series=int(cfg.get('n_series', 6)),
        head_dim=int(cfg.get('head_dim', 512)),
        dropout=float(cfg.get('dropout', 0.4)),
        encode_chunk=_cap_chunk(cfg, 2),
        lstm_hidden=int(cfg.get('lstm_hidden', 256)),
        pretrained=False,
    )
    model.load_state_dict(ckpt['model'], strict=True)
    model.to(DEVICE).eval()
    print('vit', cfg.get('run_name'), 'epoch', ckpt.get('epoch'),
          'gold', round(float(ckpt.get('gold_macro_auc')), 4))
    return model


def load_resnet(weights_path, cls, model_name, run_name, swa_label, swa_epochs):
    ckpt = torch.load(str(weights_path), map_location='cpu', weights_only=False)
    cfg = ckpt.get('config', {})
    if str(cfg.get('run_name')) != run_name:
        raise RuntimeError(cfg.get('run_name'))
    if str(cfg.get('model')) != model_name:
        raise RuntimeError(cfg.get('model'))
    if cfg.get('laterality'):
        raise RuntimeError('ce run ne doit pas etre flipped')
    _expect_swa(ckpt, swa_label, swa_epochs)
    model = cls(
        backbone=cfg.get('backbone', 'resnet50'),
        n_series=int(cfg.get('n_series', 6)),
        head_dim=int(cfg.get('head_dim', 512)),
        dropout=float(cfg.get('dropout', 0.4)),
        encode_chunk=_cap_chunk(cfg, 8),
        lstm_hidden=int(cfg.get('lstm_hidden', 256)),
        pretrained=False,
    )
    model.load_state_dict(ckpt['model'], strict=True)
    model.to(DEVICE).eval()
    print(run_name, 'epoch', ckpt.get('epoch'), 'lat', cfg.get('laterality'),
          'gold', round(float(ckpt.get('gold_macro_auc')), 4))
    return model


model_vit = load_vit(WEIGHTS_VIT)
model_v20h = load_resnet(
    WEIGHTS_V20H, KneeNetV20h, 'kneenet_v20h',
    'v20h_crop145_dino_ep11_oof_v3b_full_oaiv2_wd01', 'swa-6-7-8', [6, 7, 8],
)
model_v20b = load_resnet(
    WEIGHTS_V20B, KneeNetV20b, 'kneenet_v20b',
    'v20b_crop130_dino_ep11_oof_v3b_full_oai3', 'swa-1-2-3', [1, 2, 3],
)

def load_coat(weights_path):
    ckpt = torch.load(str(weights_path), map_location='cpu', weights_only=False)
    cfg = ckpt.get('config', {})
    if str(cfg.get('run_name')) != 'coat_a_llrd08_v3b_oofmask_full_oai3':
        raise RuntimeError(cfg.get('run_name'))
    if str(cfg.get('model')) != 'kneenet_coat_a':
        raise RuntimeError(cfg.get('model'))
    if str(cfg.get('backbone')) != 'coatnet_0_rw_224.sw_in1k':
        raise RuntimeError(cfg.get('backbone'))
    if cfg.get('laterality'):
        raise RuntimeError('coat ne doit pas etre flipped')
    if cfg.get('img_size') not in (None, 224):
        raise RuntimeError(cfg.get('img_size'))
    _expect_swa(ckpt, 'swa-1-2-3', [1, 2, 3])
    model = KneeNetCoat(
        backbone=str(cfg.get('backbone')),
        n_series=int(cfg.get('n_series', 6)),
        head_dim=int(cfg.get('head_dim', 512)),
        dropout=float(cfg.get('dropout', 0.4)),
        encode_chunk=_cap_chunk(cfg, 8),
        lstm_hidden=int(cfg.get('lstm_hidden', 256)),
        pretrained=False,
        img_size=224,
    )
    model.load_state_dict(ckpt['model'], strict=True)
    model.to(DEVICE).eval()
    print('coat', cfg.get('run_name'), 'epoch', ckpt.get('epoch'),
          'lat', cfg.get('laterality'),
          'gold', round(float(ckpt.get('gold_macro_auc')), 4))
    return model


model_coat = load_coat(WEIGHTS_COAT)

test = pd.read_csv(TEST_CSV)
test_series = pd.read_csv(TEST_SERIES_CSV)
import base64
import io

_STACK = np.load(io.BytesIO(base64.b64decode('...')))
_MU = _STACK["mu"][:, 0, :]
_SD = _STACK["sd"][:, 0, :]
_KEEP = _STACK["keep"][:, 0, :]
if [str(x) for x in _STACK["order"].tolist()] != ["ViT-k320", "v20b", "v20h", "CoAt-a"]:
    raise RuntimeError(_STACK["order"])
if str(_STACK["kind"]) != 'mlp':
    raise RuntimeError(_STACK["kind"])
if _MU.shape != (12, 34):
    raise RuntimeError(_MU.shape)


def _design_one(logit_m12, empty6):
    z = np.asarray(logit_m12, np.float32)[None]
    empty_b = np.asarray(empty6, np.float32)[None]
    m = z.shape[1]
    s = empty_b.shape[1]
    n_t = z.shape[2]
    x = np.empty((n_t, z.shape[0], m + s + m * s), np.float32)
    x[:, :, :m] = np.transpose(z, (2, 0, 1))
    x[:, :, m:m + s] = empty_b[None]
    inter = z[:, :, None, :] * empty_b[:, None, :, None]
    x[:, :, m + s:] = np.transpose(inter, (3, 0, 1, 2)).reshape(n_t, z.shape[0], m * s)
    return x[:, 0, :]


def _stack_prob(probs, empty6):
    p = np.clip(np.asarray(probs, np.float64), 1e-4, 1.0 - 1e-4)
    logit = np.log(p / (1.0 - p)).astype(np.float32)
    raw = _design_one(logit, empty6)
    xs = (raw - _MU) / _SD * _KEEP
    t = torch.from_numpy(np.ascontiguousarray(xs))
    h = torch.nn.functional.gelu(
        torch.einsum("td,tdh->th", t, torch.from_numpy(_STACK["fc1"]))
        + torch.from_numpy(_STACK["b1"])
    )
    out = torch.einsum("th,th->t", h, torch.from_numpy(_STACK["fc2"])) + torch.from_numpy(_STACK["b2"])
    prob = torch.sigmoid(out).numpy()
    if prob.shape != (12,) or not np.isfinite(prob).all():
        raise RuntimeError(prob.shape)
    return prob


print(
    'STACK_MLP', "epoch", int(_STACK["epoch"]),
    "train", round(float(_STACK["auc_train"]), 4),
    "val", round(float(_STACK["auc_val"]), 4),
    "gold_stack", round(float(_STACK["auc_gold"]), 4),
    "d", int(_MU.shape[1]), "head", 'MLP 8 unites GELU',
)
print(len(test), 'etudes | sans TTA | pas de flip | FOV vit/v20b/coat 130 mm, v20h 145 mm')
_ = (120, 0.960108)


def _forward(model, x):
    if DEVICE == 'cuda':
        with torch.autocast('cuda', dtype=AMP_DTYPE):
            return model(x)
    return model(x)


rows = []
empty_acc = np.zeros(6, np.float32)
t0 = time.time()
with torch.no_grad():
    for i, uid in enumerate(test.StudyInstanceUID):
        _SERIES_CACHE.clear()
        srows = test_series[test_series.StudyInstanceUID == uid]
        sdir = TEST_SERIES_DIR / uid
        CROP_MM = 130.0
        vol_224 = study_to_tensor(sdir, srows, n_series=6, laterality=False, img_size=224)
        vol_vit = study_to_tensor(sdir, srows, n_series=6, laterality=False, img_size=320)
        _SERIES_CACHE.clear()
        CROP_MM = 145.0
        vol_h = study_to_tensor(sdir, srows, n_series=6, laterality=False, img_size=224)
        CROP_MM = 130.0
        if vol_vit.shape[-1] != 320 or vol_224.shape[-1] != 224 or vol_h.shape[-1] != 224:
            raise RuntimeError((vol_vit.shape, vol_224.shape, vol_h.shape))
        x_224 = torch.from_numpy(vol_224).unsqueeze(0).to(DEVICE)
        x_vit = torch.from_numpy(vol_vit).unsqueeze(0).to(DEVICE)
        x_h = torch.from_numpy(vol_h).unsqueeze(0).to(DEVICE)
        p_vit = torch.sigmoid(_forward(model_vit, x_vit).float())
        p_v20b = torch.sigmoid(_forward(model_v20b, x_224).float())
        p_v20h = torch.sigmoid(_forward(model_v20h, x_h).float())
        p_coat = torch.sigmoid(_forward(model_coat, x_224).float())
        probs = np.stack([
            p_vit.detach().float().cpu().numpy().reshape(-1),
            p_v20b.detach().float().cpu().numpy().reshape(-1),
            p_v20h.detach().float().cpu().numpy().reshape(-1),
            p_coat.detach().float().cpu().numpy().reshape(-1),
        ], axis=0)
        empty = np.array([
            0.0 if _pick_protocol_row(srows, plane, fluid) is not None else 1.0
            for plane, fluid in PROTOCOL_SLOTS
        ], dtype=np.float32)
        empty_acc += empty
        prob = _stack_prob(probs, empty)
        rows.append([uid] + prob.tolist())
        if (i + 1) % 50 == 0:
            print(f'{i+1}/{len(test)} | {(time.time()-t0)/60:.1f} min', flush=True)

print('STACK_EMPTY_SUM', empty_acc.round(3).tolist())
pred = pd.DataFrame(rows, columns=['StudyInstanceUID'] + TARGETS)
pred.head()


In [ ]:
# --- submission.csv au format exact de sample_submission --------------------
# Obligatoire pour le scoring Kaggle : ecrire /kaggle/working/submission.csv
from pathlib import Path as _Out

sample = pd.read_csv(SAMPLE_SUB)
sub = pred.set_index('StudyInstanceUID').reindex(sample.StudyInstanceUID).reset_index()
sub = sub[list(sample.columns)]
prior = sub[TARGETS].mean()
sub[TARGETS] = sub[TARGETS].fillna(prior)

out_dir = _Out('/kaggle/working') if _Out('/kaggle/working').exists() else _Out('.')
out_csv = out_dir / 'submission.csv'
sub.to_csv(out_csv, index=False)
cwd_csv = _Out('submission.csv')
if cwd_csv.resolve() != out_csv.resolve():
    sub.to_csv(cwd_csv, index=False)

print('ecrit', out_csv.resolve(), '| lignes', len(sub), '| existe', out_csv.exists())
print(sub.shape, '| colonnes OK :', list(sub.columns) == list(sample.columns))
print(sub[TARGETS].describe().loc[['mean', 'std', 'min', 'max']].round(3).to_string())
assert out_csv.exists() and out_csv.stat().st_size > 0, 'submission.csv manquant'
assert not sub[TARGETS].isna().any().any(), 'NaN dans la submission !'
assert list(sub.columns) == list(sample.columns), 'colonnes != sample_submission'
sub.head()
